# 1 — Reference HVGs and SCVI/SCANVI training (D10_Lapa)

1. Intersect `var_names` across all slim files (backed reads, no counts in RAM).
2. Select HVGs on the **D10 reference only** (scArches needs queries' gene set ⊇ reference's).
3. Re-emit each slim file subset to the reference HVG list, into `INTEGRATION_DIR/slim_hvg/`.
4. Train SCVI then SCANVI on D10. Save both models.
5. Sanity assert: SCANVI recall on the reference recovers `initial_cellassign_prediction` ≥ 95%.

Set `ACCELERATOR='gpu'` and bump epochs if running on HPC.

In [ ]:
import sys, gc
from pathlib import Path

_p = Path('.').resolve()
while not (_p / 'src' / 'config.py').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.config import INTEGRATION_DIR
from src.transfer_learning import (
    intersect_var_names,
    select_reference_hvgs,
    prepare_slim_h5ad,
    train_reference,
)

import scanpy as sc
import numpy as np

ACCELERATOR = 'auto'  # set to 'gpu' on HPC
N_TOP_GENES = 3000
REFERENCE_KEY = 'D10_Lapa'
QUERY_KEYS = ['D2_DZ', 'D2_Lapa', 'D4_DZ', 'D4_Lapa']
REFERENCE_CONFIDENCE_THRESHOLD = None  # set to 0.85 if D10 ISC bias propagates

SLIM_DIR    = INTEGRATION_DIR / 'slim'
SLIM_HVG    = INTEGRATION_DIR / 'slim_hvg'
MODEL_DIR   = INTEGRATION_DIR / 'scarches_model'
SLIM_HVG.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 1. Intersect var_names across slim files (backed; no counts loaded)
all_keys = [REFERENCE_KEY] + QUERY_KEYS
common_genes = []
_common = None
for k in all_keys:
    a = sc.read_h5ad(SLIM_DIR / f'{k}.h5ad', backed='r')
    names = set(a.var_names.tolist())
    a.file.close()
    _common = names if _common is None else (_common & names)
    print(f'{k}: {len(names)} genes')
common_genes = sorted(_common)
print(f'\nintersection across all datasets: {len(common_genes)} genes')

In [ ]:
# 2. Reference HVGs — restricted to genes present in every dataset
ref_slim = sc.read_h5ad(SLIM_DIR / f'{REFERENCE_KEY}.h5ad')
ref_slim = ref_slim[:, common_genes].copy()
sc.pp.highly_variable_genes(
    ref_slim,
    n_top_genes=N_TOP_GENES,
    flavor='seurat_v3',
    layer='counts',
    subset=False,
)
hvgs = ref_slim.var_names[ref_slim.var['highly_variable']].tolist()
print(f'selected {len(hvgs)} HVGs from {REFERENCE_KEY}')
del ref_slim
gc.collect()

In [ ]:
# 3. Re-emit slim_hvg files (one dataset in RAM at a time)
for k in all_keys:
    in_path = SLIM_DIR / f'{k}.h5ad'
    out_path = SLIM_HVG / f'{k}.h5ad'
    a = sc.read_h5ad(in_path)
    keep = [g for g in hvgs if g in a.var_names]
    missing = len(hvgs) - len(keep)
    a = a[:, keep].copy()
    a.write_h5ad(out_path, compression='gzip')
    print(f'{k}: {a.n_obs} x {a.n_vars} written ({missing} HVGs missing)')
    del a
    gc.collect()

In [ ]:
# 4. Train SCVI -> SCANVI on the reference
paths = train_reference(
    reference_slim_path=SLIM_HVG / f'{REFERENCE_KEY}.h5ad',
    model_dir=MODEL_DIR,
    label_col='initial_cellassign_prediction',
    batch_key='participant',
    categorical_covariates=('Treatment',),
    n_latent=30,
    n_layers=2,
    layer='counts',
    unknown_label='Unknown',
    confidence_col='cellassign_confidence',
    reference_confidence_threshold=REFERENCE_CONFIDENCE_THRESHOLD,
    max_epochs_scvi=None,    # let scvi auto-derive
    max_epochs_scanvi=20,
    seed=0,
    accelerator=ACCELERATOR,
)
paths

In [ ]:
# 5. Sanity: SCANVI predictions on the reference recover the input labels
ref = sc.read_h5ad(paths['reference_h5ad'])
agree = (ref.obs['scanvi_prediction_ref'].astype(str) == ref.obs['initial_cellassign_prediction'].astype(str)).mean()
print(f'reference recall: {agree:.3%}')
assert agree >= 0.95, f'reference recall {agree:.3%} below 95% threshold; investigate before mapping queries'

In [ ]:
# Confusion table: input vs SCANVI prediction on reference
import pandas as pd
pd.crosstab(ref.obs['initial_cellassign_prediction'], ref.obs['scanvi_prediction_ref'])

Reference is trained and validated. Continue with `2_Map_Queries.ipynb`.